# Data Mart SQL — Solutions, Results & Business Findings

This notebook covers the nine business questions in `data_mart_solution.sql`. Each result is queried from MySQL and key findings are generated from the returned data.

**Prerequisites:** MySQL 8.0+, the original `weekly_sales` table loaded, and `clean_weekly_sales` built by running `data_mart_solution.sql`. Set `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, and `MYSQL_DATABASE` environment variables. No passwords are stored in the notebook.

## 1. Connect and validate

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

required = ["MYSQL_HOST", "MYSQL_USER", "MYSQL_PASSWORD"]
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise EnvironmentError("Set environment variables: " + ", ".join(missing))
port = os.getenv("MYSQL_PORT", "3306")
database = os.getenv("MYSQL_DATABASE", "sql_project_p2")
url = (
    f"mysql+pymysql://{os.environ['MYSQL_USER']}:{os.environ['MYSQL_PASSWORD']}"
    f"@{os.environ['MYSQL_HOST']}:{port}/{database}"
)
engine = create_engine(url, pool_pre_ping=True)

def run_query(sql):
    return pd.read_sql_query(text(sql), engine)

source_check = run_query("""
SELECT COUNT(*) AS row_count,
       MIN(week_date) AS first_week,
       MAX(week_date) AS last_week,
       COUNT(DISTINCT region) AS regions,
       COUNT(DISTINCT platform) AS platforms,
       SUM(transactions) AS total_transactions,
       SUM(sales) AS total_sales
FROM clean_weekly_sales
""")
display(source_check)
if source_check.empty or int(source_check.loc[0, "row_count"]) == 0:
    raise ValueError("clean_weekly_sales is empty. Load weekly_sales and run the preparation SQL first.")

## 2. Q1 — Missing week numbers 1–52

**Logic:** recursive sequence + NOT EXISTS. **Business use:** identify coverage gaps before publishing trends. This tests week numbers, not missing date rows per region.

In [ ]:
q1 = run_query("""
WITH RECURSIVE week_sequence AS (
    SELECT 1 AS week_number
    UNION ALL
    SELECT week_number + 1 FROM week_sequence WHERE week_number < 52
)
SELECT ws.week_number AS missing_week_number
FROM week_sequence ws
WHERE NOT EXISTS (
    SELECT 1 FROM clean_weekly_sales c
    WHERE c.week_number = ws.week_number
)
ORDER BY ws.week_number
""")
display(q1)
print("Finding: missing week numbers are " + (", ".join(q1["missing_week_number"].astype(str)) if not q1.empty else "none among 1–52") + ". Confirm whether ISO week 53 applies to this reporting calendar.")

## 3. Q2 — Annual transaction totals

**Logic:** sum transactions by calendar year. **Business use:** compare transaction volume across years.

In [ ]:
q2 = run_query("""
SELECT calendar_year, SUM(transactions) AS total_transactions
FROM clean_weekly_sales
GROUP BY calendar_year
ORDER BY calendar_year
""")
display(q2)
if len(q2) >= 2:
    first, last = q2.iloc[0], q2.iloc[-1]
    delta = last["total_transactions"] - first["total_transactions"]
    print(f"Finding: transaction volume changed by {delta:,.0f} from {int(first['calendar_year'])} to {int(last['calendar_year'])}. Check coverage and business context before interpreting this as growth.")

## 4. Q3 — Sales by year, month and region

**Logic:** group by calendar year + month + region. Including year prevents January from different years being combined.

In [ ]:
q3 = run_query("""
SELECT calendar_year, month_number, region, SUM(sales) AS total_sales
FROM clean_weekly_sales
GROUP BY calendar_year, month_number, region
ORDER BY calendar_year, month_number, total_sales DESC
""")
display(q3)
if not q3.empty:
    r = q3.loc[q3["total_sales"].idxmax()]
    print(f"Finding: the highest year-month-region total in this output is {r['region']} in {int(r['calendar_year'])}-{int(r['month_number']):02d}, with sales of {r['total_sales']:,.2f}.")

## 5. Q4 — Transaction volume by platform

In [ ]:
q4 = run_query("""
SELECT platform, SUM(transactions) AS total_transactions
FROM clean_weekly_sales
GROUP BY platform
ORDER BY total_transactions DESC
""")
display(q4)
if not q4.empty:
    r = q4.iloc[0]
    print(f"Finding: {r['platform']} has the largest recorded transaction volume ({r['total_transactions']:,.0f}). Compare sales value too; transaction volume alone does not determine revenue.")

## 6. Q5 — Monthly Retail vs Shopify sales mix

**Logic:** aggregate by year/month/platform, then divide each platform's sales by total sales for that month.

In [ ]:
q5 = run_query("""
WITH monthly_platform_sales AS (
    SELECT calendar_year, month_number, platform, SUM(sales) AS platform_sales
    FROM clean_weekly_sales
    GROUP BY calendar_year, month_number, platform
),
monthly_totals AS (
    SELECT calendar_year, month_number, SUM(platform_sales) AS all_platform_sales
    FROM monthly_platform_sales
    GROUP BY calendar_year, month_number
)
SELECT p.calendar_year, p.month_number,
       ROUND(100 * SUM(CASE WHEN p.platform='Retail' THEN p.platform_sales ELSE 0 END)
             / NULLIF(MAX(t.all_platform_sales),0), 2) AS retail_percentage,
       ROUND(100 * SUM(CASE WHEN p.platform='Shopify' THEN p.platform_sales ELSE 0 END)
             / NULLIF(MAX(t.all_platform_sales),0), 2) AS shopify_percentage
FROM monthly_platform_sales p
JOIN monthly_totals t
  ON t.calendar_year=p.calendar_year AND t.month_number=p.month_number
GROUP BY p.calendar_year, p.month_number
ORDER BY p.calendar_year, p.month_number
""")
display(q5)
print("Finding: review months where channel share shifts most; a share change can result from one platform growing, another shrinking, or both.")

## 7. Q6 — Annual sales share by demographic

In [ ]:
q6 = run_query("""
SELECT calendar_year, demographic, SUM(sales) AS demographic_sales,
       ROUND(100 * SUM(sales) /
             NULLIF(SUM(SUM(sales)) OVER (PARTITION BY calendar_year),0), 2) AS sales_percentage
FROM clean_weekly_sales
GROUP BY calendar_year, demographic
ORDER BY calendar_year, sales_percentage DESC
""")
display(q6)
if not q6.empty:
    idx = q6.groupby("calendar_year")["sales_percentage"].idxmax()
    leaders = q6.loc[idx, ["calendar_year", "demographic", "sales_percentage"]]
    print("Finding: highest sales-share demographic by year:")
    display(leaders)

## 8. Q7 — Age-band and demographic mix within Retail

In [ ]:
q7 = run_query("""
SELECT age_band, demographic, SUM(sales) AS total_sales,
       ROUND(100 * SUM(sales) / NULLIF(SUM(SUM(sales)) OVER (),0), 2) AS retail_sales_percentage
FROM clean_weekly_sales
WHERE platform='Retail'
GROUP BY age_band, demographic
ORDER BY total_sales DESC
""")
display(q7)
if not q7.empty:
    r = q7.iloc[0]
    print(f"Finding: {r['age_band']} / {r['demographic']} is the largest listed Retail segment combination ({r['total_sales']:,.2f}; {r['retail_sales_percentage']}% of Retail sales). Treat segment coding as a business assumption to validate.")

## 9. Q8 — Regional year-over-year change

**Logic:** LAG by region and calendar year. Percentage change is undefined if the prior-year value is zero or absent.

In [ ]:
q8 = run_query("""
WITH regional_yearly_sales AS (
    SELECT calendar_year, region, SUM(sales) AS yearly_sales
    FROM clean_weekly_sales
    GROUP BY calendar_year, region
),
with_previous_year AS (
    SELECT calendar_year, region, yearly_sales,
           LAG(yearly_sales) OVER (PARTITION BY region ORDER BY calendar_year) AS prior_year_sales
    FROM regional_yearly_sales
)
SELECT calendar_year, region, yearly_sales, prior_year_sales,
       yearly_sales-prior_year_sales AS sales_change,
       ROUND(100*(yearly_sales-prior_year_sales)/NULLIF(prior_year_sales,0),2) AS sales_change_percentage
FROM with_previous_year
ORDER BY calendar_year, sales_change_percentage DESC
""")
display(q8)
valid = q8.dropna(subset=["sales_change_percentage"])
if not valid.empty:
    up = valid.loc[valid["sales_change_percentage"].idxmax()]
    down = valid.loc[valid["sales_change_percentage"].idxmin()]
    print(f"Finding: largest percentage increase: {up['region']} in {int(up['calendar_year'])} ({up['sales_change_percentage']:.2f}%).")
    print(f"Finding: largest percentage decrease: {down['region']} in {int(down['calendar_year'])} ({down['sales_change_percentage']:.2f}%).")
    print("These are descriptive changes; they do not identify the cause.")

## 10. Q9 — Weighted average transaction value by customer type

**Logic:** total sales / total transactions, not a simple average of row-level averages. This gives each transaction equal weight if the source transaction counts are correctly defined.

In [ ]:
q9 = run_query("""
SELECT customer_type,
       ROUND(SUM(sales)/NULLIF(SUM(transactions),0),2) AS weighted_average_transaction_value,
       SUM(transactions) AS total_transactions,
       SUM(sales) AS total_sales
FROM clean_weekly_sales
GROUP BY customer_type
ORDER BY weighted_average_transaction_value DESC
""")
display(q9)
if not q9.empty:
    r = q9.iloc[0]
    print(f"Finding: {r['customer_type']} has the highest weighted average transaction value ({r['weighted_average_transaction_value']:,.2f}). Check transaction volume and customer mix before allocating promotions.")

## 11. Executive findings and business actions

In [ ]:
print("DATA MART SUMMARY")
if not q2.empty:
    print("Annual transaction totals:")
    display(q2)
if not q4.empty:
    print("Platform transaction mix:")
    display(q4)
if not q8.empty:
    print("Regional year-over-year movement:")
    display(q8)
if not q9.empty:
    print("Weighted average transaction value by customer type:")
    display(q9)
print("\nBusiness actions:")
print("- Resolve missing-week and calendar-coverage gaps before comparing periods.")
print("- Use platform mix and regional YoY changes to select where to investigate further.")
print("- Pair demographic sales share with segment size and transaction value.")
print("- Treat the sustainability initiative as a hypothesis, not a proven cause; use a suitable comparison design to estimate impact.")